# 01 — Explore Voting Data
**MyVoterWisdom · [github.com/sysWisdom/myvoterwisdom](https://github.com/sysWisdom/myvoterwisdom)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sysWisdom/myvoterwisdom/blob/main/notebooks/01_explore_data.ipynb)

> Non-partisan educational tool. See [DISCLAIMER.md](../DISCLAIMER.md) before use.

This notebook loads and visualizes `voting_pres_data.csv` — 233 county-year records covering
US presidential elections from 2004 to 2024 across 39 counties in 25 states.

## Step 1 — Environment Setup (Colab or Local)

In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_ROOT = None

if IN_COLAB:
    # Clone the repo if not already present
    if not os.path.exists('/content/myvoterwisdom'):
        os.system('git clone https://github.com/sysWisdom/myvoterwisdom.git /content/myvoterwisdom')
    REPO_ROOT = '/content/myvoterwisdom'
    # Install extra deps not in base Colab image
    os.system('pip install -q imbalanced-learn faiss-cpu sentence-transformers')
else:
    # Running locally — notebooks/ is one level below project root
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

sys.path.insert(0, REPO_ROOT)
DATA_PATH = os.path.join(REPO_ROOT, 'data', 'voting_pres_data.csv')
print(f"REPO_ROOT : {REPO_ROOT}")
print(f"DATA_PATH : {DATA_PATH}")
print(f"File exists: {os.path.exists(DATA_PATH)}")

## Step 2 — Load voting_pres_data.csv

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')  # utf-8-sig strips BOM present in this CSV

# Enforce correct dtypes regardless of platform (Colab vs Windows)
df['Election Year'] = df['Election Year'].astype(int)
df['Wisdom'] = df['Wisdom'].map({'True': True, 'False': False}).fillna(df['Wisdom']).astype(bool)

print(f"Shape: {df.shape}")
print(f"\nColumns (dtypes):\n{df.dtypes}")
df.head()

## Step 3 — Inspect the Dataset

In [ ]:
print("=== Summary Statistics ===")
display(df.describe(include='all'))

print("\n=== Missing Values ===")
print(df.isnull().sum())

print(f"\nElection years : {sorted(df['Election Year'].unique())}")
print(f"States         : {sorted(df['State'].unique())}")
print(f"Unique counties: {df['County'].nunique()}")

## Step 4 — Voter Turnout by Election Year

In [ ]:
yearly = df.groupby('Election Year').agg(
    Total_Voted=('Total Voted', 'sum'),
    Total_Ballots=('Total Ballots Cast', 'sum'),
    Avg_Turnout_Pct=('Total Ballots Cast', lambda x: (x / df.loc[x.index, 'Total Registered Voters']).mean() * 100)
).reset_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — total votes
ax1.bar(yearly['Election Year'].astype(str), yearly['Total_Voted'] / 1e6,
        color=sns.color_palette('muted')[0], edgecolor='white')
ax1.set_title('Total Votes Cast by Election Year\n(dataset sample — 39 counties)')
ax1.set_xlabel('Election Year')
ax1.set_ylabel('Total Votes (millions)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))

# Line trend — avg turnout %
ax2.plot(yearly['Election Year'], yearly['Avg_Turnout_Pct'], marker='o',
         color=sns.color_palette('muted')[2], linewidth=2.5, markersize=8)
ax2.fill_between(yearly['Election Year'], yearly['Avg_Turnout_Pct'], alpha=0.15,
                 color=sns.color_palette('muted')[2])
ax2.set_title('Average Turnout Rate by Election Year\n(Ballots Cast / Registered Voters)')
ax2.set_xlabel('Election Year')
ax2.set_ylabel('Avg Turnout (%)')
ax2.set_ylim(0, 100)

plt.tight_layout()
plt.show()
display(yearly)

## Step 5 — Wisdom Flag Distribution

The Wisdom column is True when a county meets ≥ 2 of these 3 conditions:
- 2020 Democratic votes > 2024 Democratic votes  
- 2020 Democratic votes > 2016 Democratic votes  
- 2020 Total ballots > 2024 Total ballots  

**Note (✅ fixed 2026-05-25):** The previous row-by-row lambda logic always returned False because it compared each row against itself. The corrected pivot-based logic in preprocess.py produces the expected distribution: **32 counties True** (2020 was a Democratic high-water mark) and **7 False** (Harris TX & Fulton GA grew Dem 2020→2024; Miami-Dade FL shifted Republican).

⚠️ **Data quality flag:** House District 40 in the dataset is a legislative district, not a county. Remove or replace this entry before training.

In [ ]:
from preprocess import compare_votes_and_ballots, update_wisdom

df_w = compare_votes_and_ballots(df.copy())
df_w = update_wisdom(df_w)

wisdom_counts = df_w['Wisdom'].value_counts()
print("Wisdom distribution:")
print(wisdom_counts)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#d73027' if v else '#4575b4' for v in wisdom_counts.index]
wisdom_counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white', rot=0)
ax.set_title('Wisdom Flag Distribution\n(True = county met ≥2/3 turnout conditions)')
ax.set_xlabel('Wisdom Value')
ax.set_ylabel('County-Year Records')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2, p.get_height()),
                ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6 — County-Level Comparison: 2016 vs 2020 vs 2024

In [ ]:
recent = df[df['Election Year'].isin([2016, 2020, 2024])].copy()
pivot = recent.pivot_table(index='County', columns='Election Year', values='Total Voted', aggfunc='sum')
pivot = pivot.dropna().sort_values(2020, ascending=True)

fig, ax = plt.subplots(figsize=(14, max(6, len(pivot) * 0.4)))
x = range(len(pivot))
width = 0.28
colors = {'2016': '#4575b4', '2020': '#74add1', '2024': '#d73027'}

for i, year in enumerate([2016, 2020, 2024]):
    if year in pivot.columns:
        offset = (i - 1) * width
        bars = ax.barh([xi + offset for xi in x], pivot[year] / 1e3,
                       height=width, label=str(year), color=colors[str(year)], edgecolor='white')

ax.set_yticks(list(x))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_xlabel('Total Votes (thousands)')
ax.set_title('Total Votes by County — 2016 vs 2020 vs 2024\n(counties present in all three years)')
ax.legend(title='Election Year')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}K'))
plt.tight_layout()
plt.show()